# Exercise 3 — Short-Term and Long-Term Agent Memory
**Completed answers + optional extensions**

**Lecture 6 · Slides 44–57 · Instructor solution**

This notebook runs independently of the other two. Run setup first. Examples use **simulated data**; no emails, web searches or payments are performed.

## Setup
Set `OPENAI_API_KEY` and optionally `OPENAI_MODEL` / `OPENAI_ENDPOINT` in your environment or `.env`. For an Azure-specific endpoint, configure an appropriate Azure client instead.

In [ ]:
%pip install -q langgraph langchain-openai langchain-core python-dotenv pydantic tenacity

In [ ]:
import os
from dotenv import load_dotenv
from typing import Annotated, Literal, TypedDict
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt, Command

load_dotenv()

# Instructor-provided model settings. OPENAI_ENDPOINT is optional for a standard OpenAI endpoint.
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
model_args = {"model": model_name, "api_key": os.getenv("OPENAI_API_KEY")}
if os.getenv("OPENAI_ENDPOINT"):
    model_args["base_url"] = os.environ["OPENAI_ENDPOINT"]

assert model_args["api_key"], "Set OPENAI_API_KEY in .env (or your environment)."
llm = ChatOpenAI(**model_args)
print("Model configured:", model_name)
# If your course uses AzureOpenAI-specific endpoints rather than an OpenAI-compatible
# /openai/v1 endpoint, ask your instructor for the appropriate client configuration.

## Part A — Conversation history and checkpoints (slides 45–51)
What an app **stores** differs from the messages it **sends to the LLM**. A LangGraph checkpointer keeps per-thread state.

In [ ]:
def learning_agent_e3(state: MessagesState):
    response = llm.invoke([
        SystemMessage(content="You are a supportive Python learning assistant. Answer concisely and use the supplied conversation context."),
        *state["messages"],
    ])
    return {"messages": [response]}

builder_e3 = StateGraph(MessagesState)
builder_e3.add_node("learning_agent", learning_agent_e3)
builder_e3.add_edge(START, "learning_agent")
app_e3 = builder_e3.compile(checkpointer=InMemorySaver())

thread_a = {"configurable": {"thread_id": "learning-chat-A"}}
thread_b = {"configurable": {"thread_id": "learning-chat-B"}}

app_e3.invoke({"messages": [HumanMessage(content="My name is Alice. I'm learning Python loops.")]}, thread_a)
response_a = app_e3.invoke({"messages": [HumanMessage(content="What topic am I learning?")]}, thread_a)
print("Same thread:", response_a["messages"][-1].content)
response_b = app_e3.invoke({"messages": [HumanMessage(content="What topic am I learning?")]}, thread_b)
print("Different thread:", response_b["messages"][-1].content)

**Reflect:** Why does thread B not automatically remember thread A? Why is a `thread_id` different from a user ID?

**Instructor answer:** Thread IDs isolate checkpoints for independent conversations. A user may have many threads; cross-thread information requires a separate store and retrieval logic.

## Part B — Managing model context (slides 47–48)
Illustrate a sliding window without deleting the stored history. For the demo, use the last N messages as a context-selection strategy (real apps should preserve any required system messages and complete tool-call pairs).

In [ ]:
history = ["I'm learning Python", "I like examples", "How do loops work?", "Show a for loop", "Explain it again"]
print("Full history:", history)
print("Recent three:", history[-3:])
print("Old messages are not deleted just because the model receives less context.")

## Part C — Long-term structured preferences (slides 52, 54–55)
Save an exact preference separately from conversation checkpoints. No embeddings or external database are required here.

In [ ]:
store_e3 = InMemoryStore()
user_id_e3 = "demo-student-alice"
namespace_e3 = ("student_preferences", user_id_e3)

# Save a structured preference — no embedding model is required.
store_e3.put(namespace_e3, "preferred_language", {"language": "Python"})

stored_e3 = store_e3.get(namespace_e3, "preferred_language")
print("Retrieved long-term preference:", stored_e3.value)

In [ ]:
def personalized_e3(state: MessagesState):
    # A real application would obtain user_id from authenticated context.
    memory = store_e3.get(namespace_e3, "preferred_language")
    preference = memory.value["language"] if memory else "unknown"
    reply = llm.invoke([
        SystemMessage(content=f"You are a programming tutor. Student's saved preferred language: {preference}."),
        *state["messages"],
    ])
    return {"messages": [reply]}

builder_e3_long = StateGraph(MessagesState)
builder_e3_long.add_node("personalized", personalized_e3)
builder_e3_long.add_edge(START, "personalized")
app_e3_long = builder_e3_long.compile(checkpointer=InMemorySaver())

new_thread_e3 = {"configurable": {"thread_id": "learning-chat-C"}}
result = app_e3_long.invoke({"messages": [HumanMessage(content="Which programming language do I prefer?")]}, new_thread_e3)
print(result["messages"][-1].content)

### Solution: Update a preference and use a new thread
Updating the same memory key overwrites its saved value. Use a fresh conversation ID to show that the preference comes from long-term storage.

In [ ]:
store_e3.put(namespace_e3, "preferred_language", {"language": "JavaScript"})
updated = app_e3_long.invoke(
    {"messages": [HumanMessage(content="Which programming language do I prefer now?")]},
    {"configurable": {"thread_id": "learning-chat-D"}},
)
print("Stored preference:", store_e3.get(namespace_e3, "preferred_language").value)
print("Agent response:", updated["messages"][-1].content)

## Part D — Memory types and retrieval (slides 46, 53, 56)
- **Semantic:** preference facts such as a preferred language.
- **Episodic:** records such as “student solved loops yesterday.”
- **Procedural:** reusable instructions such as “give a hint before revealing an answer.”
- **Vector search** can retrieve semantically similar memories but requires embeddings and a search index; the basic exercise instead uses exact structured lookup.

### ⭐ Optional challenges — worked solutions
The following examples demonstrate episodic records, user-specific namespaces, updates and deletion. This is illustrative application storage, not a general-purpose security layer.

In [ ]:
# 1. Save and retrieve an episodic memory.
from datetime import date
store_e3.put(namespace_e3, "completed_loops_exercise", {
    "event": "Completed a Python loops exercise",
    "date": date.today().isoformat(),
    "type": "episodic",
})
item = store_e3.get(namespace_e3, "completed_loops_exercise")
print("Episodic memory:", item.value)

# 2. Give a different user their own memory namespace.
bob_namespace = ("student_preferences", "demo-student-bob")
store_e3.put(bob_namespace, "preferred_language", {"language": "Java"})
print("Alice:", store_e3.get(namespace_e3, "preferred_language").value)
print("Bob:", store_e3.get(bob_namespace, "preferred_language").value)

# 3. Update and then delete a temporary preference.
store_e3.put(namespace_e3, "editor_theme", {"theme": "dark"})
print("Before update:", store_e3.get(namespace_e3, "editor_theme").value)
store_e3.put(namespace_e3, "editor_theme", {"theme": "light"})
print("After update:", store_e3.get(namespace_e3, "editor_theme").value)
store_e3.delete(namespace_e3, "editor_theme")
print("After deletion:", store_e3.get(namespace_e3, "editor_theme"))

**Teaching notes:** `InMemorySaver` and `InMemoryStore` do not survive process restarts. User namespaces are a logical organization pattern and must be paired with real access controls in a deployed application. For reliable comparison, inspect saved values directly rather than relying on exact model wording.